# 04 · Agentic RAG

这一节参考课件 `RAG_theory` 的 Agentic RAG：
- **CRAG**：先评估检索质量，不够好就纠错/补检索
- **Adaptive-RAG**：先给 query 分流（简单走轻链路，复杂走重链路）
- **MemoRAG**：把高价值中间结果沉淀成 memory，后续优先复用

本 notebook 目标是“讲清楚控制闭环”，所以实现尽量精简直观：
- 仍然复用前面写好的 Chroma collection：`autel_annual_report_2024`
- 用 LLM 做路由/评估（可替换成规则/小模型）
- 每一步都打印 trace，便于课堂讲解

> 依赖：先跑 `01_data_02_chunk_ingest.ipynb` 写入 `data/chroma`。


In [1]:
from __future__ import annotations

import hashlib
import json
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Literal
from uuid import uuid4

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client
from langfuse.langchain import CallbackHandler


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()

    for candidate in (cwd, cwd.parent):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError("未找到项目根目录，请从 RAG_project 根目录或 notebooks 目录运行本 notebook。")



def load_project_env(project_root: Path) -> Path | None:
    for env_path in (project_root / ".env", project_root.parent / ".env"):
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


PROJECT_ROOT = resolve_project_root()
ENV_FILE = load_project_env(PROJECT_ROOT)
CHROMA_DIR = PROJECT_ROOT / "data/chroma"
COLLECTION = "autel_annual_report_2024"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
embed_model = os.getenv("EMBED_MODEL")
chat_model = os.getenv("CHAT_MODEL")
langfuse_public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
langfuse_secret_key = os.getenv("LANGFUSE_SECRET_KEY")
langfuse_base_url = os.getenv("LANGFUSE_BASE_URL")

assert CHROMA_DIR.exists(), f"找不到 Chroma 目录：{CHROMA_DIR.resolve()}（先跑 01_data_02_chunk_ingest.ipynb）"
assert openai_api_key, f"未加载 OPENAI_API_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"
assert langfuse_public_key, f"未加载 LANGFUSE_PUBLIC_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"
assert langfuse_secret_key, f"未加载 LANGFUSE_SECRET_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"
assert langfuse_base_url, f"未加载 LANGFUSE_BASE_URL（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"

client_kwargs = {
    "api_key": openai_api_key,
    "base_url": openai_base_url,
}

# 当前本地 Chroma 已按 1536 维 embedding 建库；若更换 embedding 模型，请先重建 data/chroma。
emb = OpenAIEmbeddings(model=embed_model, **client_kwargs)
vs = Chroma(collection_name=COLLECTION, embedding_function=emb, persist_directory=str(CHROMA_DIR))

llm = ChatOpenAI(model=chat_model, temperature=0, **client_kwargs)
langfuse = get_client()
langfuse_handler = CallbackHandler()
LF_USER_ID = os.getenv("LANGFUSE_USER_ID", "rag-notebook-user")
LF_SESSION_ID = f"agentic-rag-{uuid4().hex[:8]}"
LF_TAGS = ["rag_project", "agentic_rag", COLLECTION]


def lf_config(step_name: str, tags: list[str] | None = None):
    merged_tags = LF_TAGS + list(tags or [])
    return {
        "callbacks": [langfuse_handler],
        "run_name": step_name,
        "metadata": {
            "langfuse_user_id": LF_USER_ID,
            "langfuse_session_id": LF_SESSION_ID,
            "langfuse_tags": merged_tags,
        },
    }


print(
    "ready:",
    COLLECTION,
    "embed:",
    embed_model,
    "chat:",
    chat_model,
    "env:",
    ENV_FILE,
    "langfuse_session:",
    LF_SESSION_ID,
)


/opt/anaconda3/envs/voc/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/var/folders/hb/4k5shxzs4c7by7lm5s0mmgrw0000gn/T/ipykernel_1277/1250904645.py:56: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vs = Chroma(collection_name=COLLECTION, embedding_function=emb, persist_directory=str(CHROMA_DIR))


ready: autel_annual_report_2024 embed: Qwen/Qwen3-Embedding-8B chat: deepseek-ai/DeepSeek-V3.2 env: /Users/mengbai/Documents/AI-training/.env


In [2]:
# 工具：向量检索（带最小 trace）


def parse_json_object(raw: str, fallback: dict):
    text = (raw or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.DOTALL).strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)

    try:
        data = json.loads(text)
        if isinstance(data, dict):
            return data
    except Exception:
        pass

    return {**fallback, "_raw": (raw or "")[:240]}


def retrieve(query: str, k: int = 5, config: dict | None = None):
    retriever = vs.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(query, config=config or {})
    return docs


def finish_langfuse():
    langfuse.flush()


def doc_dedupe_key(doc):
    return (doc.metadata.get("chunk_id", doc.metadata.get("doc_id")), doc.page_content[:80])


def show_docs(docs, max_chars: int = 260):
    for i, d in enumerate(docs, 1):
        meta = {
            k: d.metadata.get(k)
            for k in ("type", "source_collection", "parse_source", "chunk_id", "h1", "h2", "h3")
            if k in d.metadata
        }
        print(f"[{i}]", meta)
        print(d.page_content[:max_chars].replace("\n", " "))
        print()


In [3]:
# 0) Self-RAG（最小闭环）：按需检索 + 对草稿做 critique，不支撑就补检索
# 课件要点：retrieve-on-demand + critique-and-fix

need_retrieve_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Self-RAG 的路由器。判断回答这个问题是否需要检索外部知识库。\n"
            "只输出 JSON：{{\"need_retrieve\": 0|1, \"reason\": \"...\"}}。",
        ),
        ("human", "问题：{q}"),
    ]
)

critique_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Self-RAG 的 critique 模块。\n"
            "给定问题、草稿答案、以及检索证据（可能为空），判断草稿是否被证据支持。\n"
            "只输出 JSON：{{\"supported\": 0|1, \"reason\": \"...\", \"rewrite\": \"...\"}}。\n"
            "- supported=0 时给一个更利于检索的 rewrite（中文）。",
        ),
        ("human", "问题：{q}\n\n草稿：{draft}\n\n证据：\n{evidence}"),
    ]
)

answer_no_ctx_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是助手。若缺少证据就保守回答，不要编造。"),
        ("human", "问题：{q}"),
    ]
)

answer_with_ctx_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是助手。必须基于给定证据回答；证据不足则明确说不足。",
        ),
        ("human", "问题：{q}\n\n证据：\n{evidence}"),
    ]
)


def self_rag(query: str, k: int = 5):
    print("=== Self-RAG trace ===")
    print("[query]", query)

    # step1: decide retrieve or not
    raw = llm.invoke(
        need_retrieve_prompt.format_messages(q=query),
        config=lf_config("self_rag.need_retrieve", ["self_rag"]),
    ).content
    j = parse_json_object(raw, {"need_retrieve": 1, "reason": ""})
    need = 1 if int(j.get("need_retrieve", 1)) == 1 else 0
    reason = j.get("reason", "")
    if "_raw" in j:
        reason = f"bad_json: {j['_raw']}"

    print("[need_retrieve]", need)
    print("[reason]", reason)

    docs = []
    if need:
        docs = retrieve(query, k=k, config=lf_config("self_rag.retrieve", ["self_rag", "retrieval"]))
        print("[retrieve] k=", k)
        show_docs(docs)

    evidence = "\n\n".join(f"[{i}] {d.page_content[:450]}" for i, d in enumerate(docs, 1))

    # step2: draft
    if docs:
        draft = llm.invoke(
            answer_with_ctx_prompt.format_messages(q=query, evidence=evidence),
            config=lf_config("self_rag.answer_with_ctx", ["self_rag", "answer"]),
        ).content.strip()
    else:
        draft = llm.invoke(
            answer_no_ctx_prompt.format_messages(q=query),
            config=lf_config("self_rag.answer_no_ctx", ["self_rag", "answer"]),
        ).content.strip()

    print("[draft head]", draft[:220].replace("\n", " "))

    # step3: critique
    raw2 = llm.invoke(
        critique_prompt.format_messages(q=query, draft=draft, evidence=evidence),
        config=lf_config("self_rag.critique", ["self_rag", "critique"]),
    ).content
    c = parse_json_object(raw2, {"supported": 0, "reason": "", "rewrite": ""})
    supported = 1 if int(c.get("supported", 0)) == 1 else 0
    creason = c.get("reason", "")
    rewrite = c.get("rewrite", "")
    if "_raw" in c:
        creason = f"bad_json: {c['_raw']}"

    print("[critique supported]", supported)
    print("[critique reason]", creason)

    if supported:
        finish_langfuse()
        return draft, docs

    # step4: fix by rewrite + retrieve + answer again
    if not rewrite:
        rewrite = query
    print("[rewrite]", rewrite)

    docs2 = retrieve(rewrite, k=k, config=lf_config("self_rag.retrieve_retry", ["self_rag", "retrieval", "retry"]))
    print("[retrieve-2] k=", k)
    show_docs(docs2)

    evidence2 = "\n\n".join(f"[{i}] {d.page_content[:450]}" for i, d in enumerate(docs2, 1))
    final = llm.invoke(
        answer_with_ctx_prompt.format_messages(q=query, evidence=evidence2),
        config=lf_config("self_rag.final_answer", ["self_rag", "answer", "retry"]),
    ).content.strip()
    print("[final head]", final[:220].replace("\n", " "))
    finish_langfuse()
    return final, docs2


Q0 = "道通2024年年报里，主营业务/产品线的收入结构是怎样的？"
self_rag_answer, self_rag_docs = self_rag(Q0, k=5)


=== Self-RAG trace ===
[query] 道通2024年年报里，主营业务/产品线的收入结构是怎样的？
[need_retrieve] 1
[reason] 该问题询问特定公司（道通）在特定年份（2024年）的年报中披露的具体财务信息（主营业务/产品线的收入结构）。这类精确、具体且时效性强的公司财务数据通常不会包含在模型的通用知识库中，必须通过检索最新的官方文档（如年报）来获取准确信息。
[retrieve] k= 5
[1] {'parse_source': 'paddleocr_vl', 'chunk_id': 46}
<div style="text-align: center;"><img src="imgs/img_in_image_box_191_155_1030_533.jpg" alt="Image" width="70%" /></div>   道通 “Evergreen” 全球 ESG 植树活动

[2] {'parse_source': 'paddleocr_vl', 'chunk_id': 953, 'h2': '四、财务报表的编制基础', 'h3': '2、持续经营'}
√适用 □不适用   本公司不存在导致对报告期末起 12 个月内的持续经营能力产生重大疑虑的事项或情况。

[3] {'parse_source': 'paddleocr_vl', 'chunk_id': 956, 'h3': '1、遵循企业会计准则的声明'}
本公司所编制的财务报表符合企业会计准则的要求，真实、完整地反映了公司的财务状况、经营成果和现金流量等有关信息。

[4] {'parse_source': 'paddleocr_vl', 'chunk_id': 801}
重大，且确定存货可变现净值涉及重大管理层判断，我们将存货可变现净值确定为关键审计事项。

[5] {'parse_source': 'paddleocr_vl', 'chunk_id': 465, 'h2': '1、负责任供应链'}
公司致力于打造可持续供应链，在保障采购需求、及时履行约定的同时，积极推动供应商提升可持续发展水平，从供应商准入、采购、评价、赋能等多方面展开全流程管理，并有针对性地加入对供应商 ESG 风险的考量。

[draft head] 根据提供的证据，无法回答

In [4]:
# 1) CRAG（Corrective RAG）：先评估检索质量，再决定要不要补救

# 课件要点：Retrieval evaluator -> (refine/search) -> generate
# 这里做课堂版最小闭环：
# - evaluator 只输出 {label, reason, rewrite}
# - label in {correct, ambiguous, incorrect}

CRAGLabel = Literal["correct", "ambiguous", "incorrect"]

crag_eval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 CRAG 的 Retrieval Evaluator。\n"
            "输入：用户问题 + 检索到的若干证据片段。\n"
            "输出 JSON：{{\"label\": \"correct|ambiguous|incorrect\", \"reason\": \"...\", \"rewrite\": \"...\"}}。\n"
            "- correct: 证据明显能支持回答\n"
            "- ambiguous: 有点相关但不够支撑，需要补检索\n"
            "- incorrect: 基本不相关，需要重写 query 再检索\n"
            "只输出 JSON，不要多余文字。",
        ),
        ("human", "问题：{q}\n\n证据：\n{evidence}"),
    ]
)

crag_rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 CRAG 的 Query Rewrite。输出 1 条更利于企业年报检索的中文查询句。不要回答问题。",
        ),
        ("human", "原始问题：{q}\n\n评估原因：{reason}\n\n建议 rewrite：{rewrite}"),
    ]
)


def crag(query: str, k: int = 5):
    print("=== CRAG trace ===")
    print("[query]", query)

    docs = retrieve(query, k=k, config=lf_config("crag.retrieve", ["crag", "retrieval"]))
    print("[retrieve-1] k=", k)
    show_docs(docs)

    evidence = "\n\n".join(f"[{i}] {d.page_content[:350]}" for i, d in enumerate(docs, 1))
    raw = llm.invoke(
        crag_eval_prompt.format_messages(q=query, evidence=evidence),
        config=lf_config("crag.evaluate", ["crag", "evaluate"]),
    ).content

    ev = parse_json_object(raw, {"label": "ambiguous", "reason": "", "rewrite": ""})
    label = ev.get("label", "ambiguous")
    if label not in {"correct", "ambiguous", "incorrect"}:
        label = "ambiguous"
    reason = ev.get("reason", "")
    rewrite_hint = ev.get("rewrite", "")
    if "_raw" in ev:
        reason = f"bad_json: {ev['_raw']}"

    print("[eval]", label)
    print("[reason]", reason)

    if label == "correct":
        finish_langfuse()
        return docs

    # corrective action: rewrite -> retrieve again
    rewritten = llm.invoke(
        crag_rewrite_prompt.format_messages(q=query, reason=reason, rewrite=rewrite_hint),
        config=lf_config("crag.rewrite", ["crag", "rewrite"]),
    ).content.strip()
    print("[rewrite]", rewritten)

    docs2 = retrieve(rewritten, k=k, config=lf_config("crag.retrieve_retry", ["crag", "retrieval", "retry"]))
    print("[retrieve-2] k=", k)
    show_docs(docs2)
    finish_langfuse()
    return docs2


Q1 = "道通2024年年报里，主营业务/产品线的收入结构是怎样的？给出相关表格或段落。"
crag_docs = crag(Q1, k=5)


=== CRAG trace ===
[query] 道通2024年年报里，主营业务/产品线的收入结构是怎样的？给出相关表格或段落。
[retrieve-1] k= 5
[1] {'parse_source': 'paddleocr_vl', 'chunk_id': 801}
重大，且确定存货可变现净值涉及重大管理层判断，我们将存货可变现净值确定为关键审计事项。

[2] {'parse_source': 'paddleocr_vl', 'chunk_id': 953, 'h2': '四、财务报表的编制基础', 'h3': '2、持续经营'}
√适用 □不适用   本公司不存在导致对报告期末起 12 个月内的持续经营能力产生重大疑虑的事项或情况。

[3] {'parse_source': 'paddleocr_vl', 'chunk_id': 418, 'h2': '1、温室气体排放情况'}
注：盘查期限为2023年9月1日至2024年08月31日。

[4] {'parse_source': 'paddleocr_vl', 'chunk_id': 1872, 'h2': '6、分部信息', 'h3': '(1). 报告分部的确定依据与会计政策'}
✓适用 ☐不适用   公司以内部组织结构、管理要求、内部报告制度等为依据确定报告分部，并以地区分部为基础确定报告分部，分别对中国境内、北美地区、欧洲地区、其他地区等的经营业绩进行考核。

[5] {'parse_source': 'paddleocr_vl', 'chunk_id': 956, 'h3': '1、遵循企业会计准则的声明'}
本公司所编制的财务报表符合企业会计准则的要求，真实、完整地反映了公司的财务状况、经营成果和现金流量等有关信息。

[eval] incorrect
[reason] 提供的证据片段均未直接涉及道通公司2024年年报中主营业务或产品线的收入结构。证据[1]讨论审计事项，[2]讨论持续经营，[3]是盘查期限，[4]提到地区分部但未给出收入数据或产品线细节，[5]是财务报表合规性声明。这些信息无法支撑回答用户关于收入结构的具体问题。
[rewrite] 道通科技2024年年报中主营业务分产品类别的营业收入及成本构成
[retrieve-2] k= 5
[1] {'parse_source'

In [5]:
# 2) Adaptive-RAG：先判断 query 难度/类型，再决定走轻/重路径
# 课件要点：complexity-aware routing（简单 query 不要走重工作流）

Route = Literal["simple", "medium", "complex"]

route_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Adaptive-RAG 的 router。根据问题复杂度输出 JSON：{{\"route\": \"simple|medium|complex\", \"reason\": \"...\"}}\n"
            "- simple：单事实/单跳，直接检索一次即可\n"
            "- medium：需要更好的召回覆盖（建议 multi-query 或 HyDE）\n"
            "- complex：需要多步/对比/多条件，建议 agentic（例如 CRAG + 多次补检索）\n"
            "只输出 JSON。",
        ),
        ("human", "问题：{q}"),
    ]
)

multi_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 Multi-Query 生成器。输出 4 条检索 query，每条一行，不要编号。",
        ),
        ("human", "问题：{q}"),
    ]
)

hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 HyDE 模块。写一段可能出现在年报中的‘假设答案’，尽量包含可检索关键词，不要编造具体数值。",
        ),
        ("human", "问题：{q}"),
    ]
)


def adaptive_rag(query: str, k: int = 5):
    raw = llm.invoke(
        route_prompt.format_messages(q=query),
        config=lf_config("adaptive_rag.route", ["adaptive_rag", "route"]),
    ).content
    r = parse_json_object(raw, {"route": "medium", "reason": ""})
    route = r.get("route", "medium")
    if route not in {"simple", "medium", "complex"}:
        route = "medium"
    reason = r.get("reason", "")
    if "_raw" in r:
        reason = f"bad_json: {r['_raw']}"

    print("=== Adaptive-RAG trace ===")
    print("[query]", query)
    print("[route]", route)
    print("[reason]", reason)

    if route == "simple":
        docs = retrieve(query, k=k, config=lf_config("adaptive_rag.retrieve_simple", ["adaptive_rag", "retrieval", "simple"]))
        show_docs(docs)
        finish_langfuse()
        return docs

    if route == "medium":
        # 课堂版：Multi-Query + HyDE 二选一；这里都跑一遍让你对比
        queries = [
            s.strip()
            for s in llm.invoke(
                multi_prompt.format_messages(q=query),
                config=lf_config("adaptive_rag.multi_query", ["adaptive_rag", "multi_query"]),
            ).content.splitlines()
            if s.strip()
        ]
        if not queries:
            queries = [query]
        print("[multi-query]", queries)
        seen, merged = set(), []
        for q in queries:
            for d in retrieve(q, k=3, config=lf_config("adaptive_rag.retrieve_multi", ["adaptive_rag", "retrieval", "multi_query"])):
                key = doc_dedupe_key(d)
                if key not in seen:
                    merged.append(d)
                    seen.add(key)
        print("[multi-query merged]", len(merged))
        show_docs(merged[:8])

        hypo = llm.invoke(
            hyde_prompt.format_messages(q=query),
            config=lf_config("adaptive_rag.hyde", ["adaptive_rag", "hyde"]),
        ).content.strip()
        print("[hyde hypo head]", hypo[:200].replace("\n", " "))
        docs_h = retrieve(hypo, k=k, config=lf_config("adaptive_rag.retrieve_hyde", ["adaptive_rag", "retrieval", "hyde"]))
        print("[hyde hits]")
        show_docs(docs_h)
        finish_langfuse()
        return merged[:8] if merged else docs_h

    # complex -> 走 CRAG（可替换成 LangGraph 多步 agent）
    docs = crag(query, k=k)
    finish_langfuse()
    return docs


Q2 = "道通年报中，研发投入的主要方向是什么？并说明对应的业务/产品线。"
adaptive_docs = adaptive_rag(Q2, k=5)


=== Adaptive-RAG trace ===
[query] 道通年报中，研发投入的主要方向是什么？并说明对应的业务/产品线。
[route] simple
[reason] 问题明确询问道通年报中研发投入的主要方向及其对应的业务或产品线，属于单事实检索，可直接从年报相关章节（如管理层讨论与分析、研发投入明细）中一次性获取答案。
[1] {'parse_source': 'paddleocr_vl', 'chunk_id': 801}
重大，且确定存货可变现净值涉及重大管理层判断，我们将存货可变现净值确定为关键审计事项。

[2] {'parse_source': 'paddleocr_vl', 'chunk_id': 958, 'h3': '3、营业周期'}
##### √适用 □不适用   公司经营业务的营业周期较短，以 12 个月作为资产和负债的流动性划分标准。

[3] {'parse_source': 'paddleocr_vl', 'chunk_id': 1073}
售、转让、报废或发生毁损的，将尚未分配的相关递延收益余额转入资产处置当期的损益。

[4] {'parse_source': 'paddleocr_vl', 'chunk_id': 1872, 'h2': '6、分部信息', 'h3': '(1). 报告分部的确定依据与会计政策'}
✓适用 ☐不适用   公司以内部组织结构、管理要求、内部报告制度等为依据确定报告分部，并以地区分部为基础确定报告分部，分别对中国境内、北美地区、欧洲地区、其他地区等的经营业绩进行考核。

[5] {'parse_source': 'paddleocr_vl', 'chunk_id': 953, 'h2': '四、财务报表的编制基础', 'h3': '2、持续经营'}
√适用 □不适用   本公司不存在导致对报告期末起 12 个月内的持续经营能力产生重大疑虑的事项或情况。



In [ ]:
# 3) MemoRAG（课堂版）：把“已回答过的高价值线索/答案片段”存入 memory store
# 思路：
# - memory 是一个独立的向量库（Chroma collection），内容是：{question, answer_clue, citations}
# - 新问题先查 memory；若命中高相似，就直接复用线索并补少量检索

MEMO_COLLECTION = "autel_annual_report_2024_memo"
memo_vs = Chroma(
    collection_name=MEMO_COLLECTION,
    embedding_function=emb,
    persist_directory=str(CHROMA_DIR),
)

memo_clue_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 MemoRAG 的 memory writer。\n"
            "给定问题 + 证据片段，提炼 5-8 条可复用的 Answer Clues（要像索引关键词），不要编造数值。\n"
            "输出 JSON：{{\"clues\": [\"...\"], \"summary\": \"...\"}}。只输出 JSON。",
        ),
        ("human", "问题：{q}\n\n证据：\n{evidence}"),
    ]
)


def memo_write(question: str, docs, memo_id: str | None = None):
    evidence = "\n\n".join(d.page_content[:500] for d in docs[:5])
    raw = llm.invoke(
        memo_clue_prompt.format_messages(q=question, evidence=evidence),
        config=lf_config("memo_rag.memo_write", ["memo_rag", "memory_write"]),
    ).content
    j = parse_json_object(raw, {"clues": [], "summary": raw[:500]})

    clues = j.get("clues", [])
    if not isinstance(clues, list):
        clues = []
    summary = j.get("summary", raw[:500])

    content = "\n".join(["Answer clues:"] + clues + ["", "Summary:", summary])
    meta = {"type": "memo", "source_collection": COLLECTION}

    memo_id = memo_id or f"memo-{hashlib.sha1(question.encode('utf-8')).hexdigest()[:16]}"
    existing = memo_vs.get(ids=[memo_id], include=[])
    if existing and existing.get("ids"):
        memo_vs.delete(ids=[memo_id])

    memo_vs.add_texts([content], ids=[memo_id], metadatas=[meta])
    memo_vs.persist()
    return memo_id


def memo_search(question: str, k: int = 3):
    retriever = memo_vs.as_retriever(search_kwargs={"k": k})
    return retriever.invoke(question, config=lf_config("memo_rag.memo_search", ["memo_rag", "memory_search"]))


# 先用一个问题跑 CRAG，然后把结果写入 memo
seed_q = "道通2024年年报里，主营业务/产品线的收入结构是怎样的？"
seed_docs = crag(seed_q, k=5)
mid = memo_write(seed_q, seed_docs)
print("[memo saved]", mid)

# 再问一个相近问题，先查 memo
follow_q = "道通年报里各产品线收入占比/结构怎么描述？"
mem_hits = memo_search(follow_q, k=3)
print("=== MemoRAG trace ===")
print("[query]", follow_q)
print("[memo hits]")
show_docs(mem_hits, max_chars=400)

# 如果 memo 命中，再用 memo 的线索补一次轻检索（这里用 memo 文本做 query）
if mem_hits:
    memo_query = mem_hits[0].page_content
    print("[retrieve with memo clue]")
    docs = retrieve(memo_query, k=5, config=lf_config("memo_rag.retrieve_with_clue", ["memo_rag", "retrieval", "memo_clue"]))
    show_docs(docs)

finish_langfuse()


=== CRAG trace ===
[query] 道通2024年年报里，主营业务/产品线的收入结构是怎样的？
[retrieve-1] k= 5
[1] {'parse_source': 'paddleocr_vl', 'chunk_id': 46}
<div style="text-align: center;"><img src="imgs/img_in_image_box_191_155_1030_533.jpg" alt="Image" width="70%" /></div>   道通 “Evergreen” 全球 ESG 植树活动

[2] {'parse_source': 'paddleocr_vl', 'chunk_id': 953, 'h2': '四、财务报表的编制基础', 'h3': '2、持续经营'}
√适用 □不适用   本公司不存在导致对报告期末起 12 个月内的持续经营能力产生重大疑虑的事项或情况。

[3] {'parse_source': 'paddleocr_vl', 'chunk_id': 956, 'h3': '1、遵循企业会计准则的声明'}
本公司所编制的财务报表符合企业会计准则的要求，真实、完整地反映了公司的财务状况、经营成果和现金流量等有关信息。

[4] {'parse_source': 'paddleocr_vl', 'chunk_id': 801}
重大，且确定存货可变现净值涉及重大管理层判断，我们将存货可变现净值确定为关键审计事项。

[5] {'parse_source': 'paddleocr_vl', 'chunk_id': 465, 'h2': '1、负责任供应链'}
公司致力于打造可持续供应链，在保障采购需求、及时履行约定的同时，积极推动供应商提升可持续发展水平，从供应商准入、采购、评价、赋能等多方面展开全流程管理，并有针对性地加入对供应商 ESG 风险的考量。

[eval] incorrect
[reason] 提供的证据片段均与用户询问的'主营业务/产品线的收入结构'无关。证据[1]是关于ESG植树活动的图片，[2]和[3]是关于持续经营能力和财务报表合规性的声明，[4]是关于关键审计事项，[5]是关于供应链管理。没有任何证据提及收入、业务构成或产品线。



- **CRAG**：
  - 先检索一次
  - LLM 只负责当“裁判”：这批证据够不够
  - 不够就触发 rewrite + 补检索
- **Adaptive-RAG**：
  - 核心是 *routing*：简单问题不要走重链路
  - 这节用 `simple/medium/complex` 三档让逻辑一眼可见
- **MemoRAG**：
  - 把“已发现的线索”当作新的可检索资产
  - 后续相似问题先查 memo，减少重复检索
